# Solar Filament Segmentation Challenge 2026 — full pipeline

This notebook reproduces the submission end to end. It loads saved weights and does **not** retrain.

1. Data discovery
2. Pre-processing: solar disk detection and limb-darkening flattening
3. **Method A (classical):** black-hat plus darkness map → adaptive hysteresis threshold → half-contrast boundary → sunspot filter
4. **Method B (U-Net):** compact U-Net with about 1.9M parameters, trained on MAGFiLO masks at 1024 px
5. Instance grouping: fragments are linked by distance, with parameters tuned for Panoptic Quality
6. Local evaluation: PQ, SQ, RQ, Dice/IoU distributions, one-to-many and many-to-one counts
7. Submission CSV (COCO RLE)

Set the paths in the next cell. Training is done by `scripts/run.py train` (see README); the flag `RETRAIN` re-runs it here.

In [ ]:
import os, sys, json, time
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("src"))
from filseg import *
from filseg import pipeline
from dataclasses import asdict

DATA       = "data"                                   # competition folder
TRAIN_JSON = "data/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TRAIN_DIR  = "data/MAGFiLO_1.0_Kaggle_2026/train/train_images"
TEST_DIR   = "data/MAGFiLO_1.0_Kaggle_2026/test/test_images"
WEIGHTS    = "weights/unet.pt"
METHOD     = "unet" if os.path.exists(WEIGHTS) else "classical"
RETRAIN    = False
print("method:", METHOD)

## 1. Data

In [ ]:
_ = discover(DATA)

In [ ]:
coco = CocoSet(TRAIN_JSON, TRAIN_DIR)
sys.path.insert(0, os.path.abspath("scripts"))
from run import split_ids          # split by image file (some images have several annotators)
train_ids, val_ids = split_ids(coco)
print(len(train_ids), "train /", len(val_ids), "val")

## 2. Pre-processing

In [ ]:
i = val_ids[0]
img = read_image(coco.path_of[i]); flat, disk = preprocess(img)
gt = coco.instance_masks(i)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("raw H-alpha")
ax[1].imshow(flat, cmap="gray", vmin=0.6, vmax=1.2); ax[1].set_title("flattened (quiet Sun = 1)")
ax[2].imshow(flat, cmap="gray", vmin=0.6, vmax=1.2)
for m in gt: ax[2].contour(m, [0.5], colors="lime", linewidths=0.6)
ax[2].set_title(f"ground truth: {len(gt)} filaments")
for a in ax: a.axis("off")

## 3. Method A — classical

In [ ]:
cp = ClassicalParams(**json.load(open("weights/classical_params.json"))) if os.path.exists("weights/classical_params.json") else ClassicalParams()
pp_c = PostParams(**json.load(open("weights/postprocess_classical.json"))) if os.path.exists("weights/postprocess_classical.json") else PostParams()
print(cp); print(pp_c)
res_c, scores_c = pipeline.evaluate(coco, val_ids[:40], "classical", pp=pp_c, cp=cp)

## 4. Method B — U-Net

In [ ]:
if RETRAIN:
    from filseg import unet
    tr = unet.build_cache(coco, train_ids, "data/_cache1024")
    va = unet.build_cache(coco, val_ids[:20], "data/_cache1024")
    unet.train(tr, va, unet.TrainConfig(time_budget_min=180), WEIGHTS)
model = None
if os.path.exists(WEIGHTS):
    from filseg import unet
    model = unet.load_model(WEIGHTS)
    pp_u = PostParams(**json.load(open("weights/postprocess_unet.json"))) if os.path.exists("weights/postprocess_unet.json") else PostParams()
    print(pp_u)
    res_u, scores_u = pipeline.evaluate(coco, val_ids[:40], "unet", model=model, pp=pp_u)

## 5–6. Comparison and qualitative results

In [ ]:
import pandas as pd
rows = {"classical": res_c}
if model is not None: rows["unet"] = res_u
display(pd.DataFrame(rows).T)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, sc in [("classical", scores_c)] + ([("unet", scores_u)] if model is not None else []):
    ax[0].hist([v for s in sc for v in s.ious], bins=20, range=(0.5, 1), alpha=0.6, label=name)
    ax[1].hist([s.pq for s in sc if not np.isnan(s.pq)], bins=20, range=(0, 1), alpha=0.6, label=name)
ax[0].set_title("IoU of matched filaments"); ax[1].set_title("PQ per image")
for a in ax: a.legend()

In [ ]:
def show(i, method, pp):
    img = read_image(coco.path_of[i]); flat, _ = preprocess(img)
    pred = pipeline.predict_instances(img, method, model, pp, cp)
    plt.figure(figsize=(9, 9)); plt.imshow(flat, cmap="gray", vmin=0.6, vmax=1.2)
    for m in coco.instance_masks(i): plt.contour(m, [0.5], colors="lime", linewidths=0.7)
    cmap = plt.get_cmap("tab20")
    for k, m in enumerate(pred): plt.contour(m, [0.5], colors=[cmap(k % 20)], linewidths=0.9)
    plt.title(f"{image_id_from_path(coco.path_of[i])}: green = ground truth, colours = predicted instances"); plt.axis("off")

for i in val_ids[:3]:
    show(i, METHOD, pp_u if model is not None else pp_c)

## 7. Submission

In [ ]:
test_paths = list_images(TEST_DIR)
pp = pp_u if model is not None else pp_c
sub = pipeline.make_submission(test_paths, METHOD, "submission.csv", model=model, pp=pp, cp=cp)
sub.head()